# Phase 6 — Analyse des résultats : comparaison des modèles

Ce notebook charge les résultats depuis MLflow et produit :
- Tableau comparatif des 3 architectures de classification (CNN scratch / DenseNet121 / DeiT-tiny)
- Courbes d'apprentissage
- Visualisation des reconstructions AE/VAE
- Comparaison des modes de fusion multimodale

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

# MLflow doit pointer sur la BD SQLite du projet
MLFLOW_DB = ROOT / 'mlflow.db'
os.environ['MLFLOW_TRACKING_URI'] = f'sqlite:///{MLFLOW_DB}'

import mlflow
mlflow.set_tracking_uri(f'sqlite:///{MLFLOW_DB}')

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = 'triage-radiologique'
print(f'MLflow DB : {MLFLOW_DB}')
print(f'Figures   : {FIGURES_DIR}')

## 1. Chargement des runs MLflow

In [ ]:
client = mlflow.tracking.MlflowClient()

def get_runs(experiment_suffix: str) -> pd.DataFrame:
    exp_name = f'{PREFIX}-{experiment_suffix}'
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        print(f'Experimentation "{exp_name}" non trouvee dans MLflow.')
        return pd.DataFrame()
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=['start_time DESC'],
    )
    print(f'{exp_name} : {len(runs)} run(s) trouvé(s)')
    return runs

clf_runs   = get_runs('classification')
anom_runs  = get_runs('anomaly')
mm_runs    = get_runs('multimodal')

print(f'\nColonnes classification : {[c for c in clf_runs.columns if not c.startswith("tags") and clf_runs[c].notna().any()][:15] if len(clf_runs) else []}')

## 2. Tableau comparatif — Classification multi-label

In [ ]:
if len(clf_runs) > 0:
    metric_cols = [
        'metrics.test_auc_macro', 'metrics.test_f1_macro',
        'metrics.test_mcc_macro', 'metrics.test_balanced_acc_macro',
    ]
    available = [c for c in metric_cols if c in clf_runs.columns]
    param_cols = ['params.n_params', 'tags.mlflow.runName']
    avail_params = [c for c in param_cols if c in clf_runs.columns]
    
    table = clf_runs[avail_params + available].copy()
    table.columns = (
        [c.replace('tags.mlflow.', '').replace('params.', '').replace('metrics.', '') for c in table.columns]
    )
    table = table.rename(columns={'runName': 'Modele'})
    
    for col in table.columns:
        if col not in ['Modele', 'n_params']:
            table[col] = pd.to_numeric(table[col], errors='coerce').round(4)
    
    print('=== Tableau comparatif — Classification multi-label ===')
    print(table.to_string(index=False))

    # Visualisation
    if 'test_auc_macro' in table.columns and table['test_auc_macro'].notna().any():
        fig, ax = plt.subplots(figsize=(9, 4))
        x = range(len(table))
        bars = ax.bar(x, table['test_auc_macro'], color=['#3498db', '#e74c3c', '#2ecc71'][:len(table)],
                      width=0.5, edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels(table['Modele'].fillna('N/A'), rotation=15, ha='right')
        ax.set_ylabel('AUC-ROC macro')
        ax.set_title('Comparaison AUC macro — 3 architectures (ChestMNIST test set)')
        ax.set_ylim(0.5, 1.0)
        for bar, val in zip(bars, table['test_auc_macro'].fillna(0)):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / '06_classification_comparison.png', dpi=150)
        plt.show()
else:
    print('Aucun run de classification dans MLflow. Lancez train_classification.py.')

## 3. Courbes d'apprentissage

In [ ]:
if len(clf_runs) > 0:
    fig, axes = plt.subplots(1, min(len(clf_runs), 3), figsize=(15, 4), sharey=True)
    if len(clf_runs) == 1:
        axes = [axes]
    
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    for i, (_, row) in enumerate(clf_runs.iterrows()):
        if i >= 3:
            break
        run_id = row['run_id']
        run_name = row.get('tags.mlflow.runName', f'Run {i+1}')
        
        history_metrics = client.get_metric_history(run_id, 'val_loss')
        f1_history = client.get_metric_history(run_id, 'f1_macro')
        train_history = client.get_metric_history(run_id, 'train_loss')
        
        ax = axes[i]
        if history_metrics:
            steps = [m.step for m in history_metrics]
            val_losses = [m.value for m in history_metrics]
            train_losses = [m.value for m in train_history] if train_history else []
            
            ax.plot(steps, val_losses, 'o-', label='val_loss', color=colors[i])
            if train_losses:
                ax.plot(steps[:len(train_losses)], train_losses, 's--', label='train_loss',
                        color=colors[i], alpha=0.6)
            ax.set_title(run_name, fontsize=10)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'Pas de métriques', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(run_name, fontsize=10)
    
    fig.suptitle('Courbes d\'apprentissage — Train / Val Loss', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '07_learning_curves.png', dpi=150)
    plt.show()
else:
    print('Pas de runs disponibles.')

## 4. AUC par classe — meilleur modèle

In [ ]:
LABEL_NAMES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia',
]

if len(clf_runs) > 0:
    # Cherche le run avec le meilleur AUC macro
    auc_col = 'metrics.test_auc_macro'
    if auc_col in clf_runs.columns:
        best_run = clf_runs.loc[clf_runs[auc_col].astype(float).idxmax()]
        best_run_id = best_run['run_id']
        best_name = best_run.get('tags.mlflow.runName', 'Best model')
        print(f'Meilleur modele : {best_name} (AUC macro = {best_run[auc_col]:.4f})')
        
        # AUC par classe
        auc_per_class = []
        for i, label in enumerate(LABEL_NAMES):
            col = f'metrics.test_auc_{label.lower()}'
            if col in clf_runs.columns:
                val = pd.to_numeric(best_run[col], errors='coerce')
                auc_per_class.append(val if not pd.isna(val) else 0.5)
            else:
                auc_per_class.append(None)
        
        auc_per_class = [v for v in auc_per_class if v is not None]
        if auc_per_class:
            fig, ax = plt.subplots(figsize=(13, 4))
            colors = plt.cm.RdYlGn([v - 0.5 for v in auc_per_class])
            bars = ax.bar(LABEL_NAMES[:len(auc_per_class)], auc_per_class,
                          color=colors, edgecolor='white')
            ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='baseline')
            ax.set_xlabel('Pathologie')
            ax.set_ylabel('AUC-ROC')
            ax.set_title(f'AUC-ROC par classe — {best_name}')
            ax.set_xticklabels(LABEL_NAMES[:len(auc_per_class)], rotation=45, ha='right', fontsize=9)
            ax.set_ylim(0.4, 1.0)
            ax.legend()
            for bar, val in zip(bars, auc_per_class):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f'{val:.2f}', ha='center', fontsize=7)
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / '08_auc_per_class.png', dpi=150)
            plt.show()
        else:
            print('AUC par classe non disponible dans MLflow (calculer apres test set eval).')

## 5. Détection d'anomalies — Score de reconstruction

In [ ]:
if len(anom_runs) > 0:
    print('=== Résultats AE/VAE ===')
    metric_cols_anom = ['metrics.anomaly_threshold', 'metrics.test_anomaly_rate']
    available_anom = [c for c in metric_cols_anom if c in anom_runs.columns]
    
    for _, row in anom_runs.iterrows():
        name = row.get('tags.mlflow.runName', row['run_id'][:8])
        thresh = pd.to_numeric(row.get('metrics.anomaly_threshold', None), errors='coerce')
        rate = pd.to_numeric(row.get('metrics.test_anomaly_rate', None), errors='coerce')
        print(f'  {name:30s} | Seuil={thresh:.5f} | Taux anomalie={100*rate:.1f}%')
        
    # Histogramme des scores (depuis artefacts MLflow)
    print('\nPour visualiser les reconstructions, voir les artefacts MLflow :')
    print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')
else:
    print('Aucun run d\'anomalie. Lancez train_anomaly.py.')

## 6. Fusion multimodale — Comparaison des stratégies

In [ ]:
if len(mm_runs) > 0:
    print('=== Résultats fusion multimodale ===')
    metric_cols_mm = ['tags.mlflow.runName', 'metrics.test_auc_macro', 'metrics.test_f1_macro']
    available_mm = [c for c in metric_cols_mm if c in mm_runs.columns]
    
    table_mm = mm_runs[available_mm].copy()
    table_mm.columns = [c.split('.')[-1] for c in table_mm.columns]
    print(table_mm.to_string(index=False))
    
    if 'test_auc_macro' in table_mm.columns and table_mm['test_auc_macro'].notna().any():
        table_mm['test_auc_macro'] = pd.to_numeric(table_mm['test_auc_macro'], errors='coerce')
        table_mm['test_f1_macro']  = pd.to_numeric(table_mm['test_f1_macro'],  errors='coerce')
        
        fig, ax = plt.subplots(figsize=(8, 4))
        x = range(len(table_mm))
        bars = ax.bar(x, table_mm['test_auc_macro'], width=0.4,
                      color=['#3498db', '#e74c3c', '#f39c12'][:len(table_mm)], label='AUC macro')
        ax.set_xticks(x)
        ax.set_xticklabels(table_mm['runName'].fillna('?'), fontsize=10)
        ax.set_ylabel('AUC-ROC macro')
        ax.set_title('Comparaison des modes de fusion — OpenI dataset')
        ax.set_ylim(0.4, 1.0)
        for bar, val in zip(bars, table_mm['test_auc_macro'].fillna(0)):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.4f}', ha='center', fontsize=9)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / '09_multimodal_comparison.png', dpi=150)
        plt.show()
else:
    print('Aucun run multimodal. Lancez train_multimodal.py.')

## 7. Synthèse finale

In [ ]:
print('=' * 65)
print('SYNTHESE FINALE — Resultats du systeme de triage radiologique')
print('=' * 65)

print('\n--- CLASSIFICATION MULTI-LABEL (ChestMNIST, 14 classes) ---')
if len(clf_runs) > 0:
    for _, row in clf_runs.iterrows():
        name = row.get('tags.mlflow.runName', row['run_id'][:8])
        auc = pd.to_numeric(row.get('metrics.test_auc_macro', None), errors='coerce')
        f1  = pd.to_numeric(row.get('metrics.test_f1_macro',  None), errors='coerce')
        mcc = pd.to_numeric(row.get('metrics.test_mcc_macro', None), errors='coerce')
        auc_str = f'{auc:.4f}' if not pd.isna(auc) else 'N/A'
        f1_str  = f'{f1:.4f}'  if not pd.isna(f1)  else 'N/A'
        mcc_str = f'{mcc:.4f}' if not pd.isna(mcc) else 'N/A'
        print(f'  {name:30s} | AUC={auc_str} | F1={f1_str} | MCC={mcc_str}')
else:
    print('  [en attente des runs MLflow]')

print('\n--- DETECTION D\'ANOMALIES (AE/VAE, seuil p95) ---')
if len(anom_runs) > 0:
    for _, row in anom_runs.iterrows():
        name = row.get('tags.mlflow.runName', row['run_id'][:8])
        thresh = pd.to_numeric(row.get('metrics.anomaly_threshold', None), errors='coerce')
        rate   = pd.to_numeric(row.get('metrics.test_anomaly_rate', None), errors='coerce')
        t_str = f'{thresh:.5f}' if not pd.isna(thresh) else 'N/A'
        r_str = f'{100*rate:.1f}%' if not pd.isna(rate) else 'N/A'
        print(f'  {name:30s} | Seuil={t_str} | Taux anomalie={r_str}')
else:
    print('  [en attente]')

print('\n--- FUSION MULTIMODALE (OpenI, image + texte) ---')
if len(mm_runs) > 0:
    for _, row in mm_runs.iterrows():
        name = row.get('tags.mlflow.runName', row['run_id'][:8])
        auc  = pd.to_numeric(row.get('metrics.test_auc_macro', None), errors='coerce')
        auc_str = f'{auc:.4f}' if not pd.isna(auc) else 'N/A'
        print(f'  {name:30s} | AUC={auc_str}')
else:
    print('  [en attente]')

print('\n✓ Figures disponibles dans reports/figures/')
import os
figs = sorted(FIGURES_DIR.glob('*.png'))
for f in figs:
    print(f'  {f.name}')